# LangChain 大模型集成实验
## 一、实验目的
1. 掌握 LangChain 基础使用，理解 ChatModel 调用规范；
2. 掌握 .env 环境变量管理密钥的开发规范；
3. 掌握两种模型调用方式：框架封装调用、原生SDK调用；
4. 熟悉大模型流式输出。

## 二、实验环境
(todo)
- 系统：Windows 10
- 编程语言：Python 3.10
- 虚拟环境：Miniconda 
- 开发工具：VS Code、Jupyter Notebook
- 依赖库：langchain、langchain-deepseek、langchain-community、python-dotenv、openai
- 模型：DeepSeek、阿里通义千问

## 三、实验内容
### 3.1 安装依赖库
说明：Jupyter中 `!` 表示执行系统命令。

In [1]:
!pip install langchain
!pip install langchain-deepseek
!pip install langchain-community
!pip install python-dotenv

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple


### 3.2 配置环境变量
在同级目录新建 `.env` 文件，写入密钥：
DEEPSEEK_API_KEY=xxx
QWEN_API_KEY=xxx

In [2]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")
qwen_api_key = os.getenv("QWEN_API_KEY")

### 3.3 LangChain 封装调用 DeepSeek
使用官方封装包快速调用对话大模型。
TODO：可以申请自己账号,配置自己的账号，写注释

In [3]:
from langchain_deepseek import ChatDeepSeek

llm = ChatDeepSeek(
    model="deepseek-chat",
    temperature=0,
    api_key=deepseek_api_key,
)

res = llm.invoke("什么是LangChain？")
print(res)

content='这是一个关于 **LangChain** 的详细解释。\n\n简单来说，**LangChain 是一个用于开发由大语言模型（LLM）驱动的应用程序的框架**。\n\n你可以把它想象成 **LLM 应用的“乐高积木”**。它提供了一套标准化的工具和组件，让开发者能够轻松地将不同的 AI 模型、数据源和工具连接起来，构建出功能强大的应用，而无需从零开始编写复杂的集成代码。\n\n---\n\n### 核心思想：为什么需要 LangChain？\n\n单独使用一个 LLM（如 GPT-4）时，它有一些明显的局限性：\n\n1.  **知识过时**：模型的知识只截止到训练数据的时间点。\n2.  **无法访问私有数据**：它不知道你公司的内部文档、数据库或个人笔记。\n3.  **无法执行操作**：它只能生成文本，不能发送邮件、查询天气、调用 API 或操作数据库。\n4.  **缺乏记忆**：在对话中，它通常不记得你之前说过什么（除非你手动把历史对话都塞给它）。\n\n**LangChain 就是为了解决这些问题而生的。** 它通过提供一系列抽象和工具，让 LLM 能够：\n\n-   **连接外部数据**（如 PDF、网页、数据库）。\n-   **与环境交互**（如调用搜索引擎、执行代码、控制其他软件）。\n-   **拥有记忆**（记住对话上下文）。\n-   **执行复杂任务**（通过“链”或“智能体”将多个步骤组合起来）。\n\n---\n\n### LangChain 的核心组件\n\nLangChain 主要由以下几个关键模块组成：\n\n#### 1. 模型 I/O（Model I/O）\n这是与 LLM 交互的基础。\n\n-   **LLMs / Chat Models**：封装了各种 LLM 的 API，如 OpenAI 的 GPT-4、Anthropic 的 Claude、Google 的 Gemini，以及开源的 Llama 等。你可以用统一的接口调用它们。\n-   **Prompts**：管理、优化和动态构建提示词（Prompt）。包括模板（Prompt Templates）和示例选择器（Example Selectors）。\n-   **Output Parsers**：将 LLM 返回的原始文本解析成结构化的数据（如 

### 3.4 原生SDK调用通义千问（流式输出）
调用阿里百炼接口，开启思考链，流式返回数据。

In [5]:
from openai import OpenAI

client = OpenAI(
    api_key=qwen_api_key,
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)

completion = client.chat.completions.create(
    model="qwen3.6-plus",
    messages=[{"role":"user","content":"你是谁"}],
    extra_body={"enable_thinking": True},
    stream=True
)

is_answering = False
print("\n=====思考过程=====")

for chunk in completion:
    # 判断空分片
    if not chunk.choices:
        continue
    delta = chunk.choices[0].delta

    # 打印思考链
    if hasattr(delta,"reasoning_content") and delta.reasoning_content:
        print(delta.reasoning_content,end="")
    # 打印回答
    if hasattr(delta,"content") and delta.content:
        if not is_answering:
            print("\n=====完整回复=====")
            is_answering = True
        print(delta.content,end="")


=====思考过程=====
Thinking Process:

1.  **Analyze the user's input**: The user is asking "你是谁" (Who are you?).
2.  **Identify the intent**: The user wants to know my identity and who created me.
3.  **Formulate the response**:
    *   State my name: Qwen (通义千问).
    *   State my creator: Alibaba Group's Tongyi Lab (阿里巴巴集团旗下通义实验室).
    *   Keep the tone helpful, sincere, and concise.
4.  **Draft the response (Internal Monologue/Draft)**:
    *   *Draft 1*: 我是 Qwen，是由阿里巴巴集团旗下通义实验室自主研发的大语言模型。
5.  **Final Polish**: Ensure it matches the required tone and guidelines.
    *   "我是 Qwen（通义千问），是由阿里巴巴集团旗下通义实验室自主研发的大语言模型。请问有什么我可以帮你的吗？" (I am Qwen, a large language model developed by Alibaba Group's Tongyi Lab. How can I help you today?)
6.  **Output Generation**: Translate the drafted response to the requested language (Chinese).

Let's refine it to be concise and accurate.
"你好！我是 Qwen（通义千问），是由阿里巴巴集团旗下通义实验室自主研发的大语言模型。很高兴为你提供帮助，请问有什么我可以为你解答或协助的吗？" (Hello! I am Qwen (Tongyi Qianwen), a large languag

## 四、实验总结
1. 完成 LangChain 基础依赖安装，熟悉框架调用流程；
2. 使用 .env 文件安全管理密钥，规范开发流程；
3. 掌握封装调用、原生SDK调用两种大模型方式；
4. 实现流式输出，区分思考链与回答内容；
5. 解决流式分片下标越界报错，掌握生产常用容错写法。